In [1]:
import sys                                  # used to edit Python's import search path below
from pathlib import Path                    # file paths as objects, works on any OS

import geopandas as gpd                     # pandas, except every row also carries a shape
import numpy as np                          # arrays and the random number generator
import pandas as pd                         # plain tables (the manifest CSV)
import rasterio                             # read and write GeoTIFFs
from rasterio.features import rasterize     # turn vector shapes into a grid of numbers
from shapely.geometry import box            # build a rectangle from its four edges

# from notebooks.wellsight.wellsight_morphology_pipeline import pits_in_tile_geodataframe

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))
from _common import DERIV, DERIV_9T, path_for, read_layer
RNG_SEED = 42
target_folder = "OTHER_DATA"
SPLIT_FRACS = {"train": 0.70, "val": 0.15, "test": 0.15}
SRC_1M = path_for("derived") / f"{target_folder}_1m"
ANN = path_for("truth") / "annotations_proj.gpkg"
ROAD_BUFFER_M = 1.5
DRAIN_BUFFER_M = 2.0  # channels are a touch wider than the 1.5 m road half-width
ref_path = Path.cwd().parent.parent.parent/"data"/ target_folder / "derived"
ANN = path_for("truth") / "annotations_proj.gpkg"  # your drawings, reprojected to metres
REF = list(ref_path.glob("*dem*.tif"))[0]
OUT = ref_path
GRID_BLOCK_X_DIM, GRID_BLOCK_Y_DIM = 12, 12

In [2]:

with rasterio.open(REF) as dem:
    profile = dem.profile.copy()
    transform = dem.transform
    heigh_var, width_var = dem.height, dem.width
    bounds_var = dem.bounds
    crs_var = dem.crs
pit_interior_gdf = read_layer(ANN, "pit_inside")
pit_wall_gdf = read_layer(ANN, "pit_wall")
pad_gdf = read_layer(ANN, "pad")

In [3]:
shapes_wall_lst = [(g, 2) for g in pit_wall_gdf.geometry if g and not g.is_empty]
shapes_floor_lst = [(g, 1) for g in pit_interior_gdf.geometry if g and not g.is_empty]
label = rasterize(shapes_wall_lst + shapes_floor_lst, out_shape=(heigh_var, width_var), transform=transform, fill=0,dtype = "uint8", all_touched=False)
# floor_counter = int((label==1).sum())
# wall_counter = int((label == 2).sum())
# pixel_ground_area = transform.a * (-transform.e)

label_profile = profile.copy()
label_profile.update(dtype = "uint8", count = 1, nodata=255, compress = "deflate", predictor = 2)
label_path = OUT / f"labels_pit_{target_folder}.tif"
with rasterio.open(label_path, mode="w", **label_profile) as dst:
    dst.write(label, 1)


In [4]:
pad_lst = [(g, 2) for g in pad_gdf.geometry if g and not g.is_empty]
pad_mask = rasterize(pad_lst, out_shape=(heigh_var, width_var), transform=transform, fill=0,dtype = "uint8")
pad_path = OUT / f"mask_pad_{target_folder}.tif"
with rasterio.open(pad_path, "w", **label_profile) as dst:
    dst.write(pad_mask, 1)


Spatial Block Grid

In [5]:
minx, miny, maxx, maxy = bounds_var.left, bounds_var.bottom, bounds_var.right, bounds_var.top
block_width_var = (maxx - minx) / GRID_BLOCK_X_DIM
block_height_var = (maxy - miny) / GRID_BLOCK_Y_DIM
blocks_lst = []
for i in range(GRID_BLOCK_X_DIM):
    for j in range(GRID_BLOCK_Y_DIM):
        x0 = minx + i * block_width_var
        y0 = miny + j * block_height_var
        blocks_lst.append({"block_id": j * GRID_BLOCK_X_DIM + i,     # 0..143, one number per block
               "ix": i, "iy": j,               # column and row, handy when debugging
               "geometry": box(x0, y0, x0 + block_width_var, y0 + block_height_var)})  # the block itself
# blocks_gdf = gpd.GeoDataFrame(blocks_lst, crs=pits_in_tile_geodataframe.crs)
blocks_gdf = gpd.GeoDataFrame(blocks_lst, crs=crs_var)



Count Picks per block via centroid

In [6]:
pit_interior_gdf = pit_interior_gdf.copy()
pit_interior_gdf['centroid_x'] = pit_interior_gdf.geometry.centroid.x
pit_interior_gdf['centroid_y'] = pit_interior_gdf.geometry.centroid.y
centers_gdf = gpd.GeoDataFrame(pit_interior_gdf[['pit_inside_id']].copy(), geometry = pit_interior_gdf.geometry.centroid, crs = pit_interior_gdf.crs)
j_holder_gdf = gpd.sjoin(centers_gdf, blocks_gdf[["block_id", "geometry"]], how = "left", predicate="within")
pit_block_gdf = j_holder_gdf.set_index("pit_inside_id")["block_id"].to_dict()
pit_interior_gdf["block_id"]= pit_interior_gdf["pit_inside_id"].map(pit_block_gdf)
print(pit_interior_gdf.columns)
pits_per_block = pit_interior_gdf.groupby("block_id").size().rename("number_pits")
blocks_gdf = blocks_gdf.merge(pits_per_block, on="block_id", how="left")
blocks_gdf["number_pits"] = blocks_gdf["number_pits"].fillna(0).astype(int)


Index(['id', 'pit_inside_id', 'pad_id', 'geometry', 'centroid_x', 'centroid_y',
       'block_id'],
      dtype='str')


In [ ]:
pit_blocks_gdf = blocks_gdf[blocks_gdf["number_pits"] > 0][['block_id', 'number_pits']].sample(frac=1, random_state=RNG_SEED).values
print(pit_blocks_gdf)
total_pits_int = int(sum(n for _, n in pit_blocks_gdf))
targets_dict = {s: total_pits_int * f for s, f in SPLIT_FRACS.items()}
running_dict = {"train":0, "val": 0, "test": 0}
split_of_dict = {}
for bid, number_pits in pit_blocks_gdf:
    deficits = {s: targets_dict[s] - running_dict[s] for s in running_dict}
    chosen_data = max(deficits, key=deficits.get)
    split_of_dict[int(bid)] = chosen_data
    running_dict[chosen_data] += int(number_pits)
blocks_gdf['split'] = blocks_gdf["block_id"].map(split_of_dict).fillna("unused")
blocks_gdf.to_file(OUT / f"pit_blocks_{target_folder}.gpkg", driver="GPKG", layer="blocks")
pit_interior_gdf["split"] = pit_interior_gdf["block_id"].map(split_of_dict).fillna("unused")
manifest = pit_interior_gdf[["pit_inside_id", "pad_id", "block_id", "split", "centroid_x", "centroid_y"]].copy()
manifest.columns = ["pit_inside_id", "pad_id", "block_id", "split", "centroid_x", "centroid_y"]
manifest.to_csv(OUT / "pit_dataset_manifest.csv", index=False)
# print(f"Wrote pit_dataset_manifest.csv  ({len(manifest)} pits)")


roadprep

In [11]:
channels = [
    ("lrm_25",       f"lrm_25_{target_folder}_1m.tif"),
    ("lrm_5",       f"lrm_5_{target_folder}_1m.tif"),
    ("slope",       f"slope_{target_folder}_1m.tif"),
    ("tpi_05",       f"tpi_05_{target_folder}_1m.tif"),
    ("openness_pos",       f"openness_pos_{target_folder}_1m.tif"),
    ("openness_neg",       f"openness_neg_{target_folder}_1m.tif"),
    ("roughness_5",       f"roughness_5_{target_folder}_1m.tif"),
]
def main_roads_process():
    paths = [(name, SRC_1M/fname) for name, fname in channels]
    for _, p in paths:
        if not p.exists():
            raise FileNotFoundError(p)

    with rasterio.open(paths[0][1]) as r0:
        profile =r0.profile.copy
        height, width = r0.height, r0.width
        transform = r0.transform
        crs = r0.crs
